In [1]:
import numpy as np
import pandas as pd
import ast

In [2]:
stock_data = pd.read_csv("stock_data/final_engineered_stock_data.csv")
sec_data = pd.read_csv("sec_finra_data/sec_filings/engineered_8k_filings.csv")
twitter_data = pd.read_csv("twitter_data/final_stock_social_sentiment_data.csv")

In [3]:
print("Columns in Stock Data: ", stock_data.columns)
print("Columns in SEC 8K Filings Data: ", sec_data.columns)
print("Columns in Twitter Data: ", twitter_data.columns)

Columns in Stock Data:  Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'Return',
       'Volatility', 'Volume_Zscore', 'Return_Zscore', 'Manipulation_Flag'],
      dtype='object')
Columns in SEC 8K Filings Data:  Index(['stock', 'accession_number', 'sentiment_label', 'sentiment_score',
       'company_name', 'filing_date', 'report_date', 'cik', 'sic', 'items',
       'date_lag', 'risk_score', 'month', 'event_count'],
      dtype='object')
Columns in Twitter Data:  Index(['date', 'created_at', 'stock', 'clean_text', 'sentiment_label',
       'sentiment_score'],
      dtype='object')


In [4]:
stock_data.head()

,Date,Open,High,Low,Close,Volume,Ticker,Return,Volatility,Volume_Zscore,Return_Zscore,Manipulation_Flag
0,2018-01-02 00:00:00-05:00,39.986349,40.489233,39.774854,40.479832,102223600,AAPL,NaN,NaN,-0.092570,NaN,0
1,2018-01-03 00:00:00-05:00,40.543277,41.017963,40.409333,40.472778,118071600,AAPL,-0.000174,NaN,0.194878,-0.070361,0
2,2018-01-04 00:00:00-05:00,40.545623,40.764168,40.437528,40.660770,89738400,AAPL,0.004645,NaN,-0.319025,0.171144,0
3,2018-01-05 00:00:00-05:00,40.757138,41.210672,40.665491,41.123726,94640000,AAPL,0.011386,NaN,-0.230121,0.508955,0
4,2018-01-08 00:00:00-05:00,40.970982,41.267071,40.872282,40.970982,82271200,AAPL,-0.003714,NaN,-0.454464,-0.247764,0


In [5]:
sec_data.head()

,stock,accession_number,sentiment_label,sentiment_score,company_name,filing_date,report_date,cik,sic,items,date_lag,risk_score,month,event_count
0,AAPL,0000320193-18-000005,neutral,0.5,APPLE INC,2018-02-01,2018-02-01,320193,3571,['Results of Operations and Financial Conditio...,0,2,2018-02,2
1,AAPL,0000320193-18-000067,neutral,0.5,APPLE INC,2018-05-01,2018-05-01,320193,3571,['Results of Operations and Financial Conditio...,0,2,2018-05,2
2,AAPL,0000320193-18-000098,neutral,0.5,APPLE INC,2018-07-31,2018-07-31,320193,3571,['Results of Operations and Financial Conditio...,0,2,2018-07,1
3,AAPL,0000320193-18-000142,neutral,0.5,APPLE INC,2018-11-01,2018-11-01,320193,3571,['Results of Operations and Financial Conditio...,0,2,2018-11,1
4,AAPL,0000320193-19-000002,neutral,0.5,APPLE INC,2019-01-02,2019-01-02,320193,3571,['Results of Operations and Financial Conditio...,0,2,2019-01,2


In [6]:
twitter_data.head()

,date,created_at,stock,clean_text,sentiment_label,sentiment_score
0,2020-04-09,2020-04-09 23:56:58+00:00,AAPL,RT : 📽️ panel assesses the big questions $AAPL...,neutral,0.999946
1,2020-04-09,2020-04-09 23:56:51+00:00,AAPL;TSLA;TLRY;GME,$UMRX bouncing. EXTREMELY OVERSOLD 💸 $DECN $OP...,neutral,0.986001
2,2020-04-09,2020-04-09 23:55:05+00:00,AAPL,$AAPL 4h/1h Sometimes these wedges break highe...,neutral,0.989112
3,2020-04-09,2020-04-09 23:54:47+00:00,AAPL,This week's Expired Signals are now published ...,neutral,0.999993
4,2020-04-09,2020-04-09 23:54:28+00:00,AAPL,"$SPY $QQQ $VXX $AAPL $BA $MSFT Guys, I figured...",neutral,0.999967


In [7]:
twitter_data['stock_list'] = twitter_data['stock'].str.split(';')
twitter_exploded = twitter_data.explode('stock_list')
twitter_exploded['stock'] = twitter_exploded['stock_list'].str.strip()
twitter_exploded.drop(columns=['stock_list'], inplace=True)

In [8]:
stock_data['stock'] = stock_data['Ticker'].astype(str).str.strip()

In [9]:
twitter_agg = twitter_exploded.groupby(['stock', 'date']).agg({
    'sentiment_score': 'mean'
}).reset_index()

twitter_agg.rename(columns={
    'date': 'Date',
    'sentiment_score': 'twitter_sentiment'
}, inplace=True)

In [10]:
sec_agg = sec_data.groupby(['stock', 'filing_date']).agg({
    'sentiment_score': 'mean',
    'risk_score': 'mean',
    'event_count': 'sum'
}).reset_index()

sec_agg.rename(columns={'filing_date': 'Date'}, inplace=True)

In [11]:
stock_data['Date'] = pd.to_datetime(stock_data['Date'], utc=True).dt.tz_localize(None).dt.date
sec_agg['Date'] = pd.to_datetime(sec_agg['Date'], utc=True).dt.tz_localize(None).dt.date
twitter_agg['Date'] = pd.to_datetime(twitter_agg['Date'], utc=True).dt.tz_localize(None).dt.date

In [12]:
print("\nStock data unique keys:")
print("Total unique stock-date pairs in stock_data:", stock_data[['stock', 'Date']].drop_duplicates().shape[0])

print("\nSEC data unique keys:")
print("Total unique stock-date pairs in sec_agg:", sec_agg[['stock', 'Date']].drop_duplicates().shape[0])

print("\nTwitter data unique keys:")
print("Total unique stock-date pairs in twitter_agg:", twitter_agg[['stock', 'Date']].drop_duplicates().shape[0])


print("Stock data date range:", stock_data['Date'].min(), "to", stock_data['Date'].max())
print("SEC data date range:", sec_agg['Date'].min(), "to", sec_agg['Date'].max())
print("Twitter data date range:", twitter_agg['Date'].min(), "to", twitter_agg['Date'].max())

stocks_in_stock = set(stock_data['stock'].unique())
stocks_in_sec = set(sec_agg['stock'].unique())
stocks_in_twitter = set(twitter_agg['stock'].unique())

print("Stocks in stock_data:", stocks_in_stock)
print("Stocks in sec_agg:", stocks_in_sec)
print("Stocks in twitter_agg:", stocks_in_twitter)

print("\nStocks present in all three datasets:", stocks_in_stock & stocks_in_sec & stocks_in_twitter)


Stock data unique keys:
Total unique stock-date pairs in stock_data: 12296

SEC data unique keys:
Total unique stock-date pairs in sec_agg: 431

Twitter data unique keys:
Total unique stock-date pairs in twitter_agg: 1324
Stock data date range: 2018-01-02 to 2023-12-29
SEC data date range: 2018-01-03 to 2023-12-21
Twitter data date range: 2020-04-09 to 2022-09-29
Stocks in stock_data: {'AMC', 'GME', 'SNDL', 'TLRY', 'GOEV', 'AAPL', 'ZOM', 'TSLA', 'PLUG'}
Stocks in sec_agg: {'GME', 'TLRY', 'AAPL', 'TSLA', 'PLUG'}
Stocks in twitter_agg: {'NAKD', 'AMC', 'GME', 'SNDL', 'TLRY', 'AAPL', 'ZOM', 'TSLA', 'PLUG'}

Stocks present in all three datasets: {'GME', 'TLRY', 'AAPL', 'TSLA', 'PLUG'}


In [13]:
combined_df = stock_data.merge(sec_agg, how='left', on=['stock', 'Date'])
combined_df = combined_df.merge(twitter_agg, how='left', on=['stock', 'Date'])

combined_df['sentiment_score'] = combined_df['sentiment_score'].fillna(0.5)  # Neutral sentiment
combined_df['risk_score'] = combined_df['risk_score'].fillna(1)              # Low risk
combined_df['event_count'] = combined_df['event_count'].fillna(0)            # No events
combined_df['twitter_sentiment'] = combined_df['twitter_sentiment'].fillna(0.5)  # Neutral sentiment

In [14]:
combined_df['Return'] = combined_df['Return'].fillna(0)
combined_df['Volatility'] = combined_df.groupby('stock')['Volatility'].transform(
    lambda x: x.fillna(x.median())
)
combined_df['Return_Zscore'] = combined_df['Return_Zscore'].fillna(0)

combined_df

,Date,Open,High,Low,Close,Volume,Ticker,Return,Volatility,Volume_Zscore,Return_Zscore,Manipulation_Flag,stock,sentiment_score,risk_score,event_count,twitter_sentiment
0,2018-01-02,39.986349,40.489233,39.774854,40.479832,102223600,AAPL,0.000000,0.014841,-0.092570,0.000000,0,AAPL,0.5,1.0,0.0,0.5
1,2018-01-03,40.543277,41.017963,40.409333,40.472778,118071600,AAPL,-0.000174,0.014841,0.194878,-0.070361,0,AAPL,0.5,1.0,0.0,0.5
2,2018-01-04,40.545623,40.764168,40.437528,40.660770,89738400,AAPL,0.004645,0.014841,-0.319025,0.171144,0,AAPL,0.5,1.0,0.0,0.5
3,2018-01-05,40.757138,41.210672,40.665491,41.123726,94640000,AAPL,0.011386,0.014841,-0.230121,0.508955,0,AAPL,0.5,1.0,0.0,0.5
4,2018-01-08,40.970982,41.267071,40.872282,40.970982,82271200,AAPL,-0.003714,0.014841,-0.454464,-0.247764,0,AAPL,0.5,1.0,0.0,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12291,2023-12-22,0.211000,0.220000,0.210000,0.217600,4682395,ZOM,0.003690,0.076343,-0.300652,0.032187,0,ZOM,0.5,1.0,0.0,0.5
12292,2023-12-26,0.216800,0.228500,0.214500,0.221900,9352873,ZOM,0.019761,0.072518,-0.210634,0.240736,0,ZOM,0.5,1.0,0.0,0.5
12293,2023-12-27,0.222500,0.228400,0.211000,0.215000,5095105,ZOM,-0.031095,0.078142,-0.292697,-0.419208,0,ZOM,0.5,1.0,0.0,0.5
12294,2023-12-28,0.216000,0.218200,0.207000,0.210000,4325280,ZOM,-0.023256,0.066950,-0.307534,-0.317481,0,ZOM,0.5,1.0,0.0,0.5


In [15]:
print(combined_df[['sentiment_score', 'risk_score', 'event_count', 'twitter_sentiment']].describe())


       sentiment_score    risk_score   event_count  twitter_sentiment
count     12296.000000  12296.000000  12296.000000       12296.000000
mean          0.500277      1.040033      0.086288           0.533846
std           0.005659      0.251406      0.563456           0.121253
min           0.500000      1.000000      0.000000           0.500000
25%           0.500000      1.000000      0.000000           0.500000
50%           0.500000      1.000000      0.000000           0.500000
75%           0.500000      1.000000      0.000000           0.500000
max           0.663636      3.000000     24.000000           0.999998


In [16]:
combined_df.to_csv("final_combined_data.csv", index=False)